# Civ5 VP Autoplay Report (R / ggplot2 edition)

Pretty `ggplot2` versions of selected charts from `Report.ipynb`. All plots use the **`hrbrthemes::theme_ipsum`** look-and-feel for a clean, editorial feel, and a shared victory-type colour palette that matches the Python report.

Outputs are written to `output/r_plots/`.

In [ ]:
suppressPackageStartupMessages({
    library(readr)
    library(dplyr)
    library(tidyr)
    library(ggplot2)
    library(scales)
    library(forcats)
    library(stringr)
    library(hrbrthemes)
    library(patchwork)
    library(ggbeeswarm)
    library(ggrepel)
})

# =============================================================================
# SYNTHETIC: Master flag for synthetic-data augmentation.
# When TRUE, bootstrap-resample the loaded data frames so that each one has
# at least SYNTH_TARGET_N rows of game-level data (game_result, religion_choices
# remapped per game, power_ranking counts scaled).  Used for previewing how
# the report will look with a larger sample size before more autoplay games
# have been logged.  Set to FALSE for production.
# To remove later, search for "SYNTHETIC:" and delete every marked block.
# =============================================================================
use_synthetic_data <- TRUE
SYNTH_TARGET_N <- 161
SYNTH_SEED <- 42

OUTPUT_DIR <- file.path("output", "r_plots")
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)

# Victory-type palette mirroring the Python report's vtc_lut.
vtc_lut <- c(
    "Cultural"          = "#ff40ff",
    "Diplomatic"        = "#6600cc",
    "Science"           = "#86f9fe",
    "Time"              = "#1a1a1a",
    "Domination"        = "#e60000",
    "Pseudo-Domination" = "#ff9900",
    "Losses"            = "#9e9e9e"
)
VICTORY_LEVELS <- c("Cultural", "Science", "Domination", "Diplomatic", "Time")

# Shared theme: hrbrthemes ipsum with a few opinionated tweaks.
theme_report <- function(base_size = 13) {
    theme_ipsum(base_size = base_size, axis_title_size = base_size,
                grid = "XY", ticks = TRUE) +
        theme(
            plot.title         = element_text(face = "bold", size = base_size + 4),
            plot.subtitle      = element_text(color = "grey35", size = base_size),
            plot.caption       = element_text(color = "grey45", size = base_size - 3),
            plot.title.position = "plot",
            legend.position    = "top",
            legend.title       = element_text(face = "bold"),
            panel.grid.minor   = element_blank()
        )
}

save_plot <- function(p, name, width = 10, height = 7, dpi = 150) {
    path <- file.path(OUTPUT_DIR, paste0(name, ".png"))
    ggsave(path, p, width = width, height = height, dpi = dpi, bg = "white")
    cat("saved:", path, "\n")
    invisible(p)
}

## Load data

We use the Spark intermediate CSVs written by the Scala log processor:

* `game_result` — one row per finished game (winner, victory type, finishing turn).
* `power_ranking` — per-civ aggregate counts of wins by victory type and overall winrate.

In [ ]:
INTERMEDIATE_CSVS <- file.path("..", "data", "MP_AUTOPLAY_VP_5_2_3",
                               "intermediate_csvs")

load_spark_csv <- function(name) {
    dir <- file.path(INTERMEDIATE_CSVS, name)
    files <- list.files(dir, pattern = "^part-.*\\.csv$", full.names = TRUE)
    stopifnot(length(files) >= 1)
    read_csv(files, show_col_types = FALSE, progress = FALSE)
}

game_result_df <- load_spark_csv("game_result") %>%
    mutate(victory_type = factor(victory_type, levels = VICTORY_LEVELS))

power_ranking_df <- load_spark_csv("power_ranking")

# -----------------------------------------------------------------------------
# SYNTHETIC: bootstrap-resample game_result_df + scale power_ranking_df counts.
# A "random sampler" (sample with replacement + light Gaussian jitter on `turn`)
# stands in for a learned auto-encoder; fast and faithful enough for layout
# previews.  The synthetic game_ids and the resampling indices are stashed in
# `.synth_game_idx` so cell #VSC-102d37b5 can remap religion_choices_df with
# the same mapping.  Remove this block when the flag is no longer needed.
# -----------------------------------------------------------------------------
if (use_synthetic_data) {
    set.seed(SYNTH_SEED)
    n_real <- nrow(game_result_df)
    n_synth <- max(0, SYNTH_TARGET_N - n_real)
    if (n_synth > 0) {
        synth_idx <- sample.int(n_real, n_synth, replace = TRUE)
        synth_games <- game_result_df[synth_idx, ] %>%
            mutate(turn = pmax(1, round(turn + rnorm(dplyr::n(), 0, 5))))
        # Make synthetic game_ids unique so downstream joins still work.
        synth_games$game_id <- paste0("synth_", seq_len(n_synth))
        # Stash mapping for the religion_choices augmentation step.
        .synth_src_game_ids <<- game_result_df$game_id[synth_idx]
        .synth_new_game_ids <<- synth_games$game_id
        .synth_new_civs     <<- as.character(synth_games$victory_civ)
        game_result_df <- bind_rows(game_result_df, synth_games)

        # Scale power_ranking aggregate counts proportionally so per-civ wins
        # and games-played reflect the new total.  Round to keep them integer.
        scale_factor <- nrow(game_result_df) / n_real
        pr_count_cols <- intersect(
            c("count_games", "wins",
              "culture_victories", "science_victories",
              "domination_victories", "diplomatic_victories", "time_victories"),
            names(power_ranking_df)
        )
        power_ranking_df <- power_ranking_df %>%
            mutate(across(all_of(pr_count_cols),
                          ~ round(suppressWarnings(as.numeric(as.character(.))) * scale_factor)))
    }
    cat("SYNTHETIC: game_result_df rows ->", nrow(game_result_df), "\n")
}
# -----------------------------------------------------------------------------

n_games <- nrow(game_result_df)
cat("games:", n_games, " civs:", n_distinct(power_ranking_df$civ), "\n")
head(game_result_df)

## 1. Victory mix — donut + violin

**Left:** donut chart showing the share of finished games won by each victory type.

**Right:** violin plot of finishing turn distributions per victory type. A second variant overlays the raw observations as a beeswarm.

In [ ]:
victory_counts <- game_result_df %>%
    count(victory_type, .drop = FALSE, name = "games") %>%
    filter(games > 0) %>%
    mutate(
        # Same factor levels as the violin plot to the right -> identical
        # category order across the two panels.
        victory_type = factor(victory_type, levels = VICTORY_LEVELS),
        share = games / sum(games),
        label = sprintf("%d  (%s)", games, percent(share, 1))
    )

# Vertical bar chart (replaces the previous donut) so the x-axis category
# order matches the violin plot in the right panel one-to-one.
donut_plot <- ggplot(victory_counts,
                     aes(x = victory_type, y = games,
                         fill = victory_type)) +
    geom_col(width = 0.78, color = "white", linewidth = 0.4,
             show.legend = FALSE) +
    geom_text(aes(label = label, color = victory_type),
              vjust = -0.4, size = 4.2, fontface = "bold",
              show.legend = FALSE) +
    scale_fill_manual(values = vtc_lut, drop = FALSE) +
    scale_color_manual(values = vtc_lut, drop = FALSE, guide = "none") +
    scale_y_continuous(expand = expansion(mult = c(0, 0.18))) +
    labs(title = "Victory Type Counts", x = NULL, y = "Games") +
    theme_report(base_size = 12) +
    theme(legend.position = "none",
          panel.grid.major.x = element_blank())

In [ ]:
build_violin <- function(with_bees = FALSE) {
    p <- ggplot(game_result_df,
                aes(x = victory_type, y = turn, fill = victory_type)) +
        geom_violin(trim = FALSE, alpha = 0.85, color = "white",
                    linewidth = 0.45, scale = "width")

    if (with_bees) {
        # Highly-visible beeswarm dots: filled circles with black outline.
        p <- p +
            geom_quasirandom(aes(fill = victory_type),
                             shape = 21, color = "black", stroke = 0.6,
                             width = 0.28, size = 2.4, alpha = 0.95,
                             show.legend = FALSE)
    }
    # No box-plot or median point overlay (per request).

    p +
        scale_fill_manual(values = vtc_lut, drop = FALSE, guide = "none") +
        scale_color_manual(values = vtc_lut, drop = FALSE) +
        scale_y_continuous(labels = comma) +
        labs(title = "Victory Time Spread",
             x = NULL,
             y = "Game-ending turn") +
        theme_report(base_size = 12) +
        theme(legend.position = "none")
}

violin_plot <- build_violin(with_bees = FALSE)
violin_bee_plot <- build_violin(with_bees = TRUE)

In [ ]:
caption_text <- sprintf("%d completed Civ5 VP autoplay games  -  Emperor difficulty", n_games)

annotation_theme <- theme_report(base_size = 13) +
    theme(
        plot.caption = element_text(color = "grey25", size = 11,
                                    hjust = 1, face = "italic")
    )

# Equal-width panels: bar chart on the left, violin on the right, with
# matching victory-type x-axis order.
combined <- (donut_plot | violin_plot) +
    plot_layout(widths = c(1, 1)) +
    plot_annotation(caption = caption_text, theme = annotation_theme)

combined_bee <- (donut_plot | violin_bee_plot) +
    plot_layout(widths = c(1, 1)) +
    plot_annotation(caption = caption_text, theme = annotation_theme)

save_plot(combined,     "01_victory_mix_donut_violin",      width = 14, height = 7)
save_plot(combined_bee, "01b_victory_mix_donut_violin_bees", width = 14, height = 7)

## 2. Win-rates by civilization

Vertical bar chart of each civilization's win count, **stacked by victory type** so the bar height reads as the overall win-rate (per-civ wins / games-played).
Bars are ordered by total win rate, descending.

In [ ]:
vtype_cols <- c(
    "culture_victories",
    "science_victories",
    "domination_victories",
    "diplomatic_victories",
    "time_victories"
)

winrate_long <- power_ranking_df %>%
    mutate(across(all_of(vtype_cols),
                  ~ suppressWarnings(as.numeric(as.character(.))) %>% replace_na(0))) %>%
    mutate(
        winrate     = as.numeric(winrate),
        count_games = as.numeric(count_games),
        total_wins  = rowSums(across(all_of(vtype_cols)))
    ) %>%
    pivot_longer(cols = all_of(vtype_cols),
                 names_to  = "vtype_raw",
                 values_to = "wins") %>%
    mutate(
        victory_type = factor(
            recode(vtype_raw,
                   culture_victories    = "Cultural",
                   science_victories    = "Science",
                   domination_victories = "Domination",
                   diplomatic_victories = "Diplomatic",
                   time_victories       = "Time"),
            levels = VICTORY_LEVELS),
        share_winrate = ifelse(count_games > 0, wins / count_games, 0)
    )

civ_order <- winrate_long %>%
    group_by(civ) %>%
    summarise(total_winrate = sum(share_winrate), .groups = "drop") %>%
    arrange(desc(total_winrate)) %>%
    pull(civ)

winrate_long <- winrate_long %>%
    mutate(civ = factor(civ, levels = civ_order))

totals_df <- winrate_long %>%
    group_by(civ) %>%
    summarise(total_winrate = sum(share_winrate),
              total_wins = sum(wins),
              .groups = "drop")

In [ ]:
winrate_plot <- ggplot(winrate_long,
                       aes(x = civ, y = share_winrate,
                           fill = victory_type)) +
    geom_col(width = 0.78, color = "white", linewidth = 0.25) +
    geom_hline(yintercept = 0.125, linetype = "dashed",
               color = "grey25", linewidth = 0.6) +
    annotate("text", x = Inf, y = 0.125,
             label = "average win rate (12.5%)",
             hjust = 1.05, vjust = -0.5,
             size = 3.4, color = "grey20", fontface = "italic") +
    geom_text(data = totals_df,
              aes(x = civ, y = total_winrate,
                  label = percent(total_winrate, accuracy = 1)),
              inherit.aes = FALSE,
              vjust = -0.4, size = 3, color = "grey25") +
    scale_fill_manual(values = vtc_lut, name = "Victory type",
                      drop = FALSE) +
    scale_y_continuous(labels = percent_format(accuracy = 1),
                       expand = expansion(mult = c(0, 0.10))) +
    labs(
        title = "Win Rate By Civilization",
        x     = "Civilization",
        y     = "Win rate"
    ) +
    theme_report(base_size = 12) +
    theme(
        axis.text.x      = element_text(angle = 45, hjust = 1, vjust = 1,
                                        size = 9),
        panel.grid.major.x = element_blank()
    )

save_plot(winrate_plot, "02_winrate_by_civ_stacked_bars", width = 14, height = 7)

## 3. Religion attainment times

Gaussian KDE of the turn at which each religion event occurred (pantheon founded,
religion founded, religion enhanced, religion reformed). Vertical solid lines
mark the per-event mean; dashed lines mark the median.

In [ ]:
religion_choices_df <- load_spark_csv("religion_choices") %>%
    left_join(game_result_df %>% select(game_id, civ = victory_civ, victory_type),
              by = c("game_id", "civ")) %>%
    mutate(win = ifelse(!is.na(victory_type), 1L, 0L))

# -----------------------------------------------------------------------------
# SYNTHETIC: clone religion_choices for each synthetic game using the mapping
# stashed in cell #VSC-f03e1251.  Each synthetic game's choices are copied from
# the real source game it was resampled from, with game_id (and `civ` if the
# winning civ is referenced) rewritten so per-game grouping works.  Remove
# along with the other SYNTHETIC blocks.
# -----------------------------------------------------------------------------
if (use_synthetic_data && exists(".synth_new_game_ids") &&
    length(.synth_new_game_ids) > 0) {
    synth_choices_list <- lapply(seq_along(.synth_new_game_ids), function(i) {
        rc <- religion_choices_df %>% filter(game_id == .synth_src_game_ids[i])
        if (nrow(rc) == 0) return(NULL)
        rc %>% mutate(game_id = .synth_new_game_ids[i])
    })
    synth_choices <- bind_rows(synth_choices_list)
    religion_choices_df <- bind_rows(religion_choices_df, synth_choices)
    cat("SYNTHETIC: religion_choices_df rows ->", nrow(religion_choices_df), "\n")
}
# -----------------------------------------------------------------------------

religion_stats_df <- load_spark_csv("religion_stats") %>%
    mutate(
        winrate = ifelse(chosen_count > 0, wins / chosen_count, 0),
        losses  = chosen_count - wins
    )

# Derive belief category (PANTHEON / FOUNDER / FOLLOWER / ENHANCER / REFORMATION)
# from the event types each belief appears under in religion_choices.
belief_categories <- religion_choices_df %>%
    distinct(belief, type) %>%
    group_by(belief) %>%
    summarise(types = paste(sort(unique(type)), collapse = "|"),
              .groups = "drop") %>%
    mutate(category = case_when(
        types == "pantheon"                              ~ "PANTHEON",
        types == "religion_reformed"                     ~ "REFORMATION",
        types == "religion_founded"                      ~ "FOUNDER",
        types == "religion_enhanced"                     ~ "ENHANCER",
        types == "religion_enhanced|religion_founded"    ~ "FOLLOWER",
        TRUE                                             ~ "OTHER"
    )) %>%
    select(belief, category)

# Palette for event types and belief categories.
religion_event_lut <- c(
    "pantheon"          = "#1f77b4",
    "religion_founded"  = "#d62728",
    "religion_enhanced" = "#ff7f0e",
    "religion_reformed" = "#2ca02c"
)
religion_event_labels <- c(
    "pantheon"          = "Pantheon Founded",
    "religion_founded"  = "Religion Founded",
    "religion_enhanced" = "Religion Enhanced",
    "religion_reformed" = "Religion Reformed"
)

belief_category_lut <- c(
    "PANTHEON"    = "#1f77b4",
    "FOUNDER"     = "#d62728",
    "ENHANCER"    = "#ff7f0e",
    "FOLLOWER"    = "#9467bd",
    "REFORMATION" = "#2ca02c"
)

cat("religion_choices rows:", nrow(religion_choices_df), "\n")
cat("beliefs:", nrow(belief_categories), "\n")
print(table(belief_categories$category))

In [ ]:
attainment_df <- religion_choices_df %>%
    distinct(game_id, civ, type, turn) %>%   # one row per event (drop duplicate beliefs)
    mutate(type = factor(type,
                         levels = c("pantheon", "religion_founded",
                                    "religion_enhanced", "religion_reformed"),
                         labels = religion_event_labels[c("pantheon",
                                                          "religion_founded",
                                                          "religion_enhanced",
                                                          "religion_reformed")]))

attainment_summary <- attainment_df %>%
    group_by(type) %>%
    summarise(mean_turn   = mean(turn),
              median_turn = median(turn),
              n           = dplyr::n(),
              .groups = "drop")

print(attainment_summary)

palette_named <- setNames(religion_event_lut[c("pantheon", "religion_founded",
                                               "religion_enhanced", "religion_reformed")],
                          religion_event_labels[c("pantheon", "religion_founded",
                                                  "religion_enhanced", "religion_reformed")])

# One annotation per (event, statistic), text identifies which is which.
annot_df <- bind_rows(
    attainment_summary %>%
        transmute(type, x = mean_turn,
                  label = sprintf("%s\nmean = %.0f", type, mean_turn)),
    attainment_summary %>%
        transmute(type, x = median_turn,
                  label = sprintf("%s\nmedian = %.0f", type, median_turn))
)

# Per-type KDE peak so each annotation is seeded near its own curve rather
# than at the global maximum.  Blue (pantheon) is narrow/tall; the others are
# wider and flatter — seeding at the global max would float their labels far
# above the actual curves.
type_peaks <- attainment_df %>%
    group_by(type) %>%
    summarise(peak_y = max(density(turn, bw = 5)$y), .groups = "drop")

y_max_est <- max(type_peaks$peak_y)

# Seed each annotation at 95 % of its own curve's peak.
annot_df <- annot_df %>%
    left_join(type_peaks, by = "type") %>%
    mutate(y_seed = peak_y * 0.95)

x_panel <- 350

attainment_plot <- ggplot(attainment_df, aes(x = turn, color = type)) +
    geom_density(bw = 5, linewidth = 1.0, alpha = 0.6, show.legend = FALSE) +
    geom_vline(data = attainment_summary,
               aes(xintercept = mean_turn, color = type),
               linewidth = 0.4, linetype = "solid", alpha = 0.7,
               show.legend = FALSE) +
    geom_vline(data = attainment_summary,
               aes(xintercept = median_turn, color = type),
               linewidth = 0.4, linetype = "dashed", alpha = 0.7,
               show.legend = FALSE) +
    # Repelled labels: max.overlaps = Inf prevents any from being dropped.
    # Each label is seeded near its own curve's peak so all four sit close to
    # their respective distributions.
    geom_label_repel(
        data = annot_df,
        aes(x = x, y = y_seed, label = label, color = type),
        size = 3.0, lineheight = 0.95,
        fill = "white", alpha = 0.95,
        label.size = 0.4, label.r = unit(0.12, "lines"),
        max.overlaps = Inf,
        force = 10, force_pull = 1,
        box.padding = unit(0.55, "lines"),
        point.padding = unit(0.10, "lines"),
        nudge_y = y_max_est * 0.08,   # small upward bias only
        segment.color = "grey50", segment.alpha = 0.5, segment.size = 0.3,
        seed = 1, show.legend = FALSE
    ) +
    scale_color_manual(values = palette_named, guide = "none") +
    scale_x_continuous(limits = c(0, x_panel), breaks = seq(0, x_panel, 50)) +
    # Modest extra vertical room above the curves for the labels.
    scale_y_continuous(expand = expansion(mult = c(0.02, 0.45))) +
    labs(title = "Religion Attainment Times",
         x = "Turn", y = "Density") +
    theme_report(base_size = 12) +
    theme(legend.position = "none")

save_plot(attainment_plot, "03_religion_attainment_times", width = 12, height = 6)

## 4. Belief frequency and performance

For each belief group (pantheon, founder, enhancer + follower at religion-enhance
time, reformation), we plot a horizontal lollipop sorted by pick count. The point
size encodes the number of picks, the colour fill encodes the win-rate
(wins / chosen_count), and the trailing label shows the win-rate.

Belief categories are derived from `religion_choices`: a belief that appears
under both `religion_founded` and `religion_enhanced` events is a follower belief;
otherwise it is classified by the unique event type it appears under.

In [ ]:
# Pick statistics for one or more "events" (rows of religion_choices_df).
# Returns one row per (belief, category) with chosen_count, wins, winrate.
belief_pick_stats <- function(choices_df, event_types) {
    choices_df %>%
        filter(type %in% event_types) %>%
        group_by(belief) %>%
        summarise(chosen_count = dplyr::n(),
                  wins         = sum(win),
                  avg_turn     = mean(turn),
                  .groups      = "drop") %>%
        mutate(winrate = wins / chosen_count) %>%
        left_join(belief_categories, by = "belief")
}

# Generic horizontal lollipop chart of frequency + win-rate.
# All text sizes are ~1.5x the prior version (base 12 -> 18, text 3 -> 4.5);
# nudge_x = 7 (fixed data-unit offset) pushes labels right of the bigger dots.
# A long-and-narrow win-rate colour bar is shown along the bottom (centered
# on the 12.5 % expected baseline), drawn small/grey so it stays inconspicuous.
plot_belief_freq_perf <- function(stats_df,
                                  title,
                                  caption = NULL,
                                  color_by_category = FALSE,
                                  category_palette = belief_category_lut,
                                  show_category_legend = FALSE,
                                  bar_color = "#4c4c4c") {
    df <- stats_df %>%
        mutate(belief = forcats::fct_reorder(belief, chosen_count))

    p <- ggplot(df, aes(x = chosen_count, y = belief)) +
        geom_segment(aes(x = 0, xend = chosen_count, yend = belief),
                     color = "grey75", linewidth = 0.5)

    if (color_by_category) {
        cat_guide <- if (show_category_legend) "legend" else "none"
        p <- p +
            geom_point(aes(size = chosen_count, fill = category),
                       shape = 21, color = "grey25", stroke = 0.4) +
            scale_fill_manual(values = category_palette,
                              name = "Category", drop = FALSE,
                              guide = cat_guide)
    } else {
        # Limits anchored so 12.5% sits at the visual centre of the bar.
        wr_max <- max(df$winrate, 0.25, na.rm = TRUE)
        wr_lim <- c(0, max(wr_max, 0.25))
        p <- p +
            geom_point(aes(size = chosen_count, fill = winrate),
                       shape = 21, color = "grey25", stroke = 0.4) +
            scale_fill_gradient2(
                low = "#b30000", mid = "#f7f7f7",
                high = "#1a9850", midpoint = 0.125,
                limits = wr_lim,
                labels = percent_format(accuracy = 1),
                breaks = c(0, 0.125, wr_lim[2]),
                name = "Win rate",
                guide = guide_colourbar(
                    title.position = "left", title.vjust = 0.95,
                    barwidth  = unit(40, "lines"),
                    barheight = unit(0.7, "lines"),
                    frame.colour = NA,
                    ticks = FALSE
                )
            )
    }

    p <- p +
        geom_text(aes(label = sprintf("%d  (%s)", chosen_count,
                                      percent(winrate, accuracy = 1))),
                  hjust = 0, nudge_x = 7,
                  size = 4.5, color = "grey25") +
        scale_size_continuous(range = c(3, 13), guide = "none") +
        scale_x_continuous(expand = expansion(mult = c(0.02, 0.40))) +
        labs(title = title, caption = caption,
             x = "Times chosen", y = NULL) +
        theme_report(base_size = 18) +
        theme(panel.grid.major.y = element_blank())

    if (color_by_category) {
        if (show_category_legend) {
            p <- p + theme(legend.position = "top")
        } else {
            p <- p + theme(legend.position = "none")
        }
    } else {
        # Inconspicuous colour bar at the bottom: doubled in size, centred,
        # still grey-labelled with no frame so it doesn't fight the data.
        p <- p + theme(
            legend.position   = "bottom",
            legend.direction  = "horizontal",
            legend.justification = "center",
            legend.box.just      = "center",
            legend.title      = element_text(size = 14, color = "grey45",
                                             face = "plain"),
            legend.text       = element_text(size = 12, color = "grey45"),
            legend.margin     = margin(t = 4, r = 0, b = 0, l = 0),
            legend.box.margin = margin(t = -2, r = 0, b = 0, l = 0),
            legend.background = element_rect(fill = NA, color = NA),
            legend.key        = element_rect(fill = NA, color = NA)
        )
    }
    p
}

In [ ]:
pantheon_stats <- belief_pick_stats(religion_choices_df, "pantheon")

pantheon_plot <- plot_belief_freq_perf(
    pantheon_stats,
    title = "Pantheon Pick Frequency And Performance"
)

save_plot(pantheon_plot, "04_pantheon_freq_perf",
          width = 13, height = max(6, 0.48 * nrow(pantheon_stats) + 3))

In [ ]:
founder_beliefs <- belief_categories %>%
    filter(category == "FOUNDER") %>% pull(belief)

founder_stats <- belief_pick_stats(religion_choices_df, "religion_founded") %>%
    filter(belief %in% founder_beliefs)

founder_plot <- plot_belief_freq_perf(
    founder_stats,
    title = "Founder Beliefs - Frequency And Performance"
)

save_plot(founder_plot, "05_founder_beliefs_freq_perf",
          width = 13, height = max(6, 0.48 * nrow(founder_stats) + 3))

In [ ]:
enhance_stats <- belief_pick_stats(religion_choices_df, "religion_enhanced") %>%
    filter(category %in% c("ENHANCER", "FOLLOWER")) %>%
    mutate(category = factor(category, levels = c("ENHANCER", "FOLLOWER")))

enhance_plot <- plot_belief_freq_perf(
    enhance_stats,
    title = "Beliefs Picked At Religion-Enhance Time",
    color_by_category = TRUE,
    show_category_legend = TRUE
)

save_plot(enhance_plot, "06_enhance_time_beliefs_freq_perf",
          width = 13, height = max(6, 0.48 * nrow(enhance_stats) + 3))

In [ ]:
reformation_stats <- belief_pick_stats(religion_choices_df, "religion_reformed")

reformation_plot <- plot_belief_freq_perf(
    reformation_stats,
    title = "Reformation Beliefs - Frequency And Performance"
)

save_plot(reformation_plot, "07_reformation_beliefs_freq_perf",
          width = 13, height = max(6, 0.48 * nrow(reformation_stats) + 3))

## 5. Technology research time by era

Per-tech research-duration distributions across eras shown as a **ridgeline plot** (one density per era), so the relative spread and skew across eras is visible at a glance instead of being collapsed into a boxplot.

In [ ]:
tech_completion_df <- load_spark_csv("technology_completion_records") %>%
    mutate(era_n = as.integer(gridx_at_start))

# -----------------------------------------------------------------------------
# SYNTHETIC: clone tech rows for each synthetic game using the cell-4 mapping.
# -----------------------------------------------------------------------------
if (use_synthetic_data && exists(".synth_new_game_ids") &&
    length(.synth_new_game_ids) > 0) {
    synth_tech_list <- lapply(seq_along(.synth_new_game_ids), function(i) {
        rc <- tech_completion_df %>% filter(game_id == .synth_src_game_ids[i])
        if (nrow(rc) == 0) return(NULL)
        rc %>% mutate(game_id = .synth_new_game_ids[i])
    })
    tech_completion_df <- bind_rows(tech_completion_df, bind_rows(synth_tech_list))
    cat("SYNTHETIC: tech_completion_df rows ->", nrow(tech_completion_df), "
")
}
# -----------------------------------------------------------------------------

era_lut <- c("1" = "Ancient", "2" = "Classical", "3" = "Medieval",
             "4" = "Renaissance", "5" = "Industrial", "6" = "Modern",
             "7" = "Atomic", "8" = "Information", "9" = "Future")

tech_era_df <- tech_completion_df %>%
    filter(!is.na(era_n), era_n >= 1, era_n <= 9, duration > 0) %>%
    mutate(era = factor(era_lut[as.character(era_n)],
                        levels = unname(era_lut)))  # bottom-up = oldest first

library(ggridges)

tech_ridge_plot <- ggplot(tech_era_df,
                          aes(x = duration, y = era, fill = era)) +
    geom_density_ridges(scale = 2.0, rel_min_height = 0.01,
                        alpha = 0.85, color = "white",
                        linewidth = 0.4, bandwidth = 1.4) +
    scale_fill_viridis_d(option = "mako", direction = -1, end = 0.92,
                         guide = "none") +
    scale_x_continuous(limits = c(0, quantile(tech_era_df$duration, 0.99)),
                       expand = expansion(mult = c(0.01, 0.05))) +
    labs(title = "Technology Research Time by Era",
         subtitle = "One density ridge per era (oldest at bottom)",
         x = "Turns to research", y = NULL) +
    theme_report(base_size = 13) +
    theme(panel.grid.major.y = element_blank())

save_plot(tech_ridge_plot, "08_tech_research_time_by_era_ridgeline",
          width = 12, height = 7)

## 6. Era progression time

Distribution of the turn at which each civ first reached each era, shown as **violins** instead of bars/boxplots, with the per-era median marked. Civs that never reach an era are dropped from that era's violin.

In [ ]:
era_transitions_df <- load_spark_csv("era_transitions") %>%
    mutate(era = as.integer(era))

# -----------------------------------------------------------------------------
# SYNTHETIC: clone era_transitions rows for each synthetic game.
# -----------------------------------------------------------------------------
if (use_synthetic_data && exists(".synth_new_game_ids") &&
    length(.synth_new_game_ids) > 0) {
    synth_et_list <- lapply(seq_along(.synth_new_game_ids), function(i) {
        rc <- era_transitions_df %>% filter(game_id == .synth_src_game_ids[i])
        if (nrow(rc) == 0) return(NULL)
        rc %>% mutate(game_id = .synth_new_game_ids[i])
    })
    era_transitions_df <- bind_rows(era_transitions_df,
                                    bind_rows(synth_et_list))
    cat("SYNTHETIC: era_transitions_df rows ->", nrow(era_transitions_df), "
")
}
# -----------------------------------------------------------------------------

# First turn each (game, civ) entered each era.
era_entry_df <- era_transitions_df %>%
    group_by(game_id, civ, era) %>%
    summarise(turn = min(turn), .groups = "drop") %>%
    filter(era >= 2, era <= 9) %>%      # drop Ancient (everyone starts there at turn 1)
    mutate(era_name = factor(era_lut[as.character(era)],
                             levels = unname(era_lut)[2:9]))

era_summary <- era_entry_df %>%
    group_by(era_name) %>%
    summarise(median_turn = median(turn),
              n           = dplyr::n(),
              .groups     = "drop")

n_civ_games <- era_entry_df %>%
    distinct(game_id, civ) %>% nrow()

era_summary <- era_summary %>%
    mutate(reach_pct = n / n_civ_games)

era_violin_plot <- ggplot(era_entry_df,
                          aes(x = era_name, y = turn, fill = era_name)) +
    geom_violin(trim = TRUE, scale = "width", alpha = 0.85,
                color = "white", linewidth = 0.45) +
    stat_summary(fun = median, geom = "point",
                 shape = 21, size = 3.2, fill = "white",
                 color = "grey15", stroke = 0.7) +
    geom_text(data = era_summary,
              aes(x = era_name, y = median_turn,
                  label = sprintf("med %d  -  reach %s",
                                  round(median_turn),
                                  percent(reach_pct, accuracy = 1))),
              inherit.aes = FALSE,
              vjust = -1.3, size = 3.4, color = "grey25") +
    scale_fill_viridis_d(option = "mako", direction = -1, end = 0.92,
                         guide = "none") +
    scale_y_continuous(labels = comma,
                       expand = expansion(mult = c(0.02, 0.10))) +
    labs(title = "Era Progression Time",
         subtitle = "First turn each civ entered each era - white dot = median",
         x = NULL, y = "Turn") +
    theme_report(base_size = 13) +
    theme(panel.grid.major.x = element_blank())

save_plot(era_violin_plot, "09_era_progression_time_violin",
          width = 13, height = 7)

## 7. Branch adoption + performance, re-interpreted

The Python report shows three stacked bar/broken-axis figures (adoption frequency, victory-type counts per branch, per-branch result breakdown). Below are three more interpretable views of the same underlying data:

1. **Adoption lollipop coloured by win-rate** - bar height encodes pick count, fill colour encodes branch win-rate centred on the 12.5% baseline (matches the religion lollipops).
2. **Branch x victory-type win-rate heatmap** - cell colour = wins / adoptions, so high-performing branch/victory pairings pop out.
3. **Branch adoption vs win-rate scatter** - x = adoption count, y = win-rate, with a dashed 12.5% reference line; over- and under-picked high-win-rate branches are immediately visible.

In [ ]:
branch_name_lut <- c("0" = "Tradition",  "1" = "Progress",
                     "2" = "Authority", "3" = "Fealty",
                     "4" = "Statecraft","5" = "Artistry",
                     "6" = "Industry",  "7" = "Imperialism",
                     "8" = "Rationalism","9" = "Freedom",
                     "10" = "Order",   "11" = "Autocracy")

branch_stats_df <- load_spark_csv("branch_stats") %>%
    mutate(branch = as.character(branch),
           victory_type = ifelse(is.na(victory_type) | victory_type == "",
                                 "Loss", victory_type))

# -----------------------------------------------------------------------------
# SYNTHETIC: clone branch_stats rows for each synthetic game.  victory_type
# remains tied to the source game so totals stay consistent.
# -----------------------------------------------------------------------------
if (use_synthetic_data && exists(".synth_new_game_ids") &&
    length(.synth_new_game_ids) > 0) {
    synth_bs_list <- lapply(seq_along(.synth_new_game_ids), function(i) {
        rc <- branch_stats_df %>% filter(game_id == .synth_src_game_ids[i])
        if (nrow(rc) == 0) return(NULL)
        rc %>% mutate(game_id = .synth_new_game_ids[i])
    })
    branch_stats_df <- bind_rows(branch_stats_df, bind_rows(synth_bs_list))
    cat("SYNTHETIC: branch_stats_df rows ->", nrow(branch_stats_df), "
")
}
# -----------------------------------------------------------------------------

branch_summary <- branch_stats_df %>%
    mutate(branch_name = factor(branch_name_lut[branch],
                                levels = unname(branch_name_lut))) %>%
    group_by(branch_name) %>%
    summarise(adoptions = dplyr::n(),
              wins      = sum(victory_type %in% VICTORY_LEVELS),
              Cultural   = sum(victory_type == "Cultural"),
              Diplomatic = sum(victory_type == "Diplomatic"),
              Science    = sum(victory_type == "Science"),
              Time       = sum(victory_type == "Time"),
              Domination = sum(victory_type == "Domination"),
              .groups   = "drop") %>%
    mutate(winrate = wins / adoptions,
           losses  = adoptions - wins)

print(branch_summary)

In [ ]:
branch_lollipop_df <- branch_summary %>%
    mutate(branch_name = forcats::fct_reorder(branch_name, adoptions))

wr_lim <- c(0, max(branch_lollipop_df$winrate, 0.25, na.rm = TRUE))

branch_lollipop_plot <- ggplot(branch_lollipop_df,
                               aes(x = adoptions, y = branch_name)) +
    geom_segment(aes(x = 0, xend = adoptions, yend = branch_name),
                 color = "grey75", linewidth = 0.5) +
    geom_point(aes(size = adoptions, fill = winrate),
               shape = 21, color = "grey25", stroke = 0.4) +
    geom_text(aes(label = sprintf("%d  (%s)", adoptions,
                                  percent(winrate, accuracy = 1))),
              hjust = 0, nudge_x = max(branch_lollipop_df$adoptions) * 0.015,
              size = 4.5, color = "grey25") +
    scale_fill_gradient2(low = "#b30000", mid = "#f7f7f7",
                        high = "#1a9850", midpoint = 0.125,
                        limits = wr_lim,
                        labels = percent_format(accuracy = 1),
                        breaks = c(0, 0.125, wr_lim[2]),
                        name = "Win rate",
                        guide = guide_colourbar(
                            title.position = "left", title.vjust = 0.95,
                            barwidth  = unit(40, "lines"),
                            barheight = unit(0.7, "lines"),
                            frame.colour = NA, ticks = FALSE)) +
    scale_size_continuous(range = c(3, 13), guide = "none") +
    scale_x_continuous(expand = expansion(mult = c(0.02, 0.20))) +
    labs(title = "Policy Branch Adoption Frequency And Win Rate",
         x = "Times adopted", y = NULL) +
    theme_report(base_size = 14) +
    theme(panel.grid.major.y = element_blank(),
          legend.position    = "bottom",
          legend.direction   = "horizontal",
          legend.justification = "center",
          legend.title = element_text(size = 11, color = "grey45"),
          legend.text  = element_text(size = 10, color = "grey45"))

save_plot(branch_lollipop_plot, "10_branch_adoption_lollipop",
          width = 12, height = 7)

In [ ]:
branch_vic_long <- branch_summary %>%
    select(branch_name, adoptions, all_of(VICTORY_LEVELS)) %>%
    pivot_longer(cols = all_of(VICTORY_LEVELS),
                 names_to = "victory_type", values_to = "wins") %>%
    mutate(victory_type = factor(victory_type, levels = VICTORY_LEVELS),
           winrate      = wins / adoptions)

# Order branches by total winrate (best on top).
branch_total_order <- branch_summary %>%
    arrange(winrate) %>% pull(branch_name) %>% as.character()
branch_vic_long <- branch_vic_long %>%
    mutate(branch_name = factor(branch_name, levels = branch_total_order))

heat_max <- max(branch_vic_long$winrate, 0.05, na.rm = TRUE)

branch_heatmap_plot <- ggplot(branch_vic_long,
                              aes(x = victory_type, y = branch_name,
                                  fill = winrate)) +
    geom_tile(color = "white", linewidth = 0.6) +
    geom_text(aes(label = ifelse(wins > 0,
                                 sprintf("%s
(%d)",
                                         percent(winrate, accuracy = 1),
                                         wins),
                                 "")),
              size = 3.4, color = "grey15", lineheight = 0.95) +
    scale_fill_gradient(low = "#f7f7f7", high = "#1a9850",
                        limits = c(0, heat_max),
                        labels = percent_format(accuracy = 1),
                        name = "Win rate",
                        guide = guide_colourbar(
                            barwidth  = unit(20, "lines"),
                            barheight = unit(0.7, "lines"),
                            frame.colour = NA, ticks = FALSE)) +
    labs(title = "Branch x Victory Type Win Rate",
         subtitle = "Cell = wins / adoptions for that (branch, victory) pair",
         x = NULL, y = NULL) +
    theme_report(base_size = 13) +
    theme(legend.position = "bottom",
          panel.grid = element_blank(),
          axis.text.x = element_text(angle = 0))

save_plot(branch_heatmap_plot, "11_branch_x_victory_heatmap",
          width = 11, height = 8)

In [ ]:
mean_winrate <- 0.125

branch_scatter_plot <- ggplot(branch_summary,
                              aes(x = adoptions, y = winrate)) +
    geom_hline(yintercept = mean_winrate, linetype = "dashed",
               color = "grey25", linewidth = 0.5) +
    annotate("text", x = Inf, y = mean_winrate,
             label = "average win rate (12.5%)",
             hjust = 1.05, vjust = -0.5,
             size = 3.4, color = "grey20", fontface = "italic") +
    geom_point(aes(size = adoptions, fill = winrate),
               shape = 21, color = "grey25", stroke = 0.5) +
    ggrepel::geom_text_repel(aes(label = branch_name),
                             size = 4.2, color = "grey20",
                             max.overlaps = Inf, seed = 7,
                             box.padding = 0.5) +
    scale_fill_gradient2(low = "#b30000", mid = "#f7f7f7",
                        high = "#1a9850", midpoint = 0.125,
                        labels = percent_format(accuracy = 1),
                        name = "Win rate",
                        guide = guide_colourbar(
                            barwidth  = unit(20, "lines"),
                            barheight = unit(0.7, "lines"),
                            frame.colour = NA, ticks = FALSE)) +
    scale_size_continuous(range = c(3, 12), guide = "none") +
    scale_y_continuous(labels = percent_format(accuracy = 1),
                       expand = expansion(mult = c(0.05, 0.10))) +
    scale_x_continuous(expand = expansion(mult = c(0.05, 0.10))) +
    labs(title = "Branch Adoption vs Win Rate",
         subtitle = "Top-left = under-adopted high-winrate branches; top-right = staples",
         x = "Times adopted", y = "Win rate") +
    theme_report(base_size = 13) +
    theme(legend.position = "bottom")

save_plot(branch_scatter_plot, "12_branch_adoption_vs_winrate_scatter",
          width = 11, height = 8)

In [ ]:
fs <- list.files(OUTPUT_DIR, recursive = TRUE, full.names = FALSE)
cat("Generated", length(fs), "file(s):\n")
cat(paste(" -", fs), sep = "\n")